# Chapter 14.3: reading circulars with retrieval

Five synthetic State S circulars, split by clause. Each chunk keeps its state, number, issue date,
effective date and language. The reader answers with the clause quoted, the number and the file, or says
it found nothing. It never writes to the rate table. TF-IDF stands in for a search service here.

In [ ]:
import re
from pathlib import Path
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
chunks = []
for path in sorted(Path('../data/circulars').glob('*.txt')):
    head, body = path.read_text().split('\n\n', 1)
    meta = dict(re.findall(r'^(\w[\w ]*?): (.+)$', head, flags=re.M))
    number = re.search(r'No\. (\S+)', head).group(1)
    for clause in re.findall(r'^(\d+)\. (.+)$', body, flags=re.M):
        chunks.append(dict(file=path.name, number=number, issued=meta['Issued'], effective=meta['Effective'],
                           language=meta['Language'], subject=meta['Subject'], clause=clause[0], text=clause[1],
                           superseded_from=None))
chunks = pd.DataFrame(chunks)
# Clause 2 of S/EXC/2026/01 is amended by S/EXC/2026/09 from 1 October 2026: mark it, do not delete it.
chunks.loc[(chunks.number == 'S/EXC/2026/01') & (chunks.clause == '2'), 'superseded_from'] = '1 October 2026 by S/EXC/2026/09'
print(len(chunks), 'clauses from', chunks.file.nunique(), 'circulars')
chunks[['number', 'clause', 'effective', 'superseded_from', 'text']]

In [ ]:
# Score on the clause text. The subject lines join the vocabulary used to spot a question no circular covers.
STOP = list(ENGLISH_STOP_WORDS | {'does', 'rate', 'percent', 'says', 'tell'})
vec = TfidfVectorizer(stop_words=STOP).fit(pd.concat([chunks.text, chunks.subject.drop_duplicates()]))
matrix = vec.transform(chunks.text)
analyse = vec.build_analyzer()

def ask(question, threshold=0.2):
    unknown = [w for w in analyse(question) if w not in vec.vocabulary_]
    if unknown:
        return f'No clause found: no circular mentions {", ".join(unknown)}. Ask a person to read the source.'
    scores = cosine_similarity(vec.transform([question]), matrix)[0]
    best = scores.argmax()
    if scores[best] < threshold:
        return 'No clause found. Ask a person to read the source.'
    c = chunks.iloc[best]
    note = f' Superseded from {c.superseded_from}.' if c.superseded_from else ''
    return f'"{c.text}" ({c.number}, clause {c.clause}, effective {c.effective}, file {c.file}).{note}'

for q in ['When does the amendment on strong beer come into force?',
          'What is the transport pass fee a bulk litre?',
          'Duty on strong beer above 5.0 percent',
          'What is the excise rate on cider and perry?']:
    print('Q:', q); print('A:', ask(q)); print()